In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
%matplotlib qt

# Параметры сетки и схемы
GRIDWIDTH  = 5
GRIDHEIGHT = 5
alpha = 0.20          # шаг по времени (для явной схемы в 2D держите <= 0.25)
iterCount = 1000      # число шагов
interval_ms = 500      # ~20 FPS

# Поле: 3x3 единицы в центре, остальное нули
u = np.zeros((GRIDHEIGHT, GRIDWIDTH), dtype=np.float32)
cy, cx = GRIDHEIGHT // 2, GRIDWIDTH // 2
u[cy-1:cy+2, cx-1:cx+2] = 1.0

# Дискретный лапласиан с периодическими границами (тор)
def laplacian(a: np.ndarray) -> np.ndarray:
    return (np.roll(a,  1, axis=0) + np.roll(a, -1, axis=0) +
            np.roll(a,  1, axis=1) + np.roll(a, -1, axis=1) - 4*a)

# Визуализация
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(u, cmap='cool', vmin=0.0, vmax=1.0, interpolation='nearest')
ax.set_title('Diffusion (periodic boundaries)')
ax.set_xticks([]); ax.set_yticks([])

# Шаг эволюции
def step(_):
    u[:] = u + alpha * laplacian(u)   # in-place обновление
    # np.clip(u, 0.0, 1.0, out=u)     # опционально, если захотите страховать значения
    im.set_data(u)
    return [im]

anim = FuncAnimation(fig, step, frames=iterCount, interval=interval_ms, blit=True)
plt.show()


# Диффузия двух веществ

# Инициализация клеток

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Параметры области и схемы
N = 16
H, W = N, 2 * N
alpha = 0.2
steps = 200

speed = 1.0
base_interval = 50
interval_ms = int(base_interval / speed)

block_size = 4

u1 = np.zeros((H, W), dtype=float)
u2 = np.zeros((H, W), dtype=float)

# Центры квадратов
cy = H // 2
r0 = cy - block_size // 2
r1 = r0 + block_size

cx_left = N // 2
c0_left = cx_left - block_size // 2
c1_left = c0_left + block_size
u1[r0:r1, c0_left:c1_left] = 1.0

cx_right = N + N // 2
c0_right = cx_right - block_size // 2
c1_right = c0_right + block_size
u2[r0:r1, c0_right:c1_right] = 1.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Параметры области и схемы
N = 16                                 # базовый размер по высоте (и половине ширины)
H, W = N, 2 * N                        # высота H и ширина W прямоугольной решётки
alpha = 0.2                            # коэффициент диффузии-схемы (D * dt / h^2)
steps = 200                            # количество шагов анимации

speed = 1.0                            # множитель скорости анимации (1.0 — базовая)
base_interval = 50                     # базовый интервал между кадрами в миллисекундах
interval_ms = int(base_interval / speed)  # реальный интервал между кадрами с учётом speed

block_size = 4                         # размер квадрата с веществом

u1 = np.zeros((H, W), dtype=float)     # поле концентраций первого вещества
u2 = np.zeros((H, W), dtype=float)     # поле концентраций второго вещества

# Центры квадратов
cy = H // 2                            # вертикальный центр области
r0 = cy - block_size // 2              # верхняя граница квадрата по вертикали
r1 = r0 + block_size                   # нижняя граница квадрата по вертикали

cx_left = N // 2                       # горизонтальный центр левой половины области
c0_left = cx_left - block_size // 2    # левая граница левого квадрата
c1_left = c0_left + block_size         # правая граница левого квадрата
u1[r0:r1, c0_left:c1_left] = 1.0       # заполняем левый квадрат первым веществом

cx_right = N + N // 2                  # горизонтальный центр правой половины области
c0_right = cx_right - block_size // 2  # левая граница правого квадрата
c1_right = c0_right + block_size       # правая граница правого квадрата
u2[r0:r1, c0_right:c1_right] = 1.0     # заполняем правый квадрат вторым веществом

In [ ]:

def log_scale(u, eps=1e-6, k=100.0):
    u_clip = np.clip(u, 0.0, None)
    v = np.log1p(k * u_clip) / np.log1p(k)
    return v

def make_rgb(u1, u2):
    rgb = np.zeros((H, W, 3), dtype=float)
    rgb[..., 0] = log_scale(u1)  # красный канал — вещество 1
    rgb[..., 2] = log_scale(u2)  # синий канал — вещество 2
    return rgb

%matplotlib notebook

fig, ax = plt.subplots()
img = make_rgb(u1, u2)
im = ax.imshow(img, interpolation="nearest")
ax.set_title("Diffusion of two substances in a box (step 0)")
ax.set_xticks([])
ax.set_yticks([])
plt.show()


In [ ]:


def diffusion_step_box(u, alpha):
    """
    Один шаг диффузии в замкнутом ящике (без потока через границы).
    Условия Неймана: производная по нормали на границе = 0.
    Реализуем через зеркальное расширение массива (mode='edge').
    """
    # Добавляем по одному "призрачному" слою с каждой стороны,
    # где значения на границе просто копируются.
    u_p = np.pad(u, pad_width=1, mode='edge')

    center = u_p[1:-1, 1:-1]
    up     = u_p[:-2, 1:-1]
    down   = u_p[2:, 1:-1]
    left   = u_p[1:-1, :-2]
    right  = u_p[1:-1, 2:]

    laplacian = up + down + left + right - 4 * center
    return u + alpha * laplacian


fig, ax = plt.subplots()
img = make_rgb(u1, u2)
im = ax.imshow(img, interpolation="nearest")
ax.set_title("Diffusion of two substances in a box (step 0)")
ax.set_xticks([])
ax.set_yticks([])

def update(frame):
    global u1, u2
    u1 = diffusion_step_box(u1, alpha)
    u2 = diffusion_step_box(u2, alpha)
    img = make_rgb(u1, u2)
    im.set_data(img)
    ax.set_title(f"Diffusion of two substances in a box (step {frame})")
    return [im]

ani = FuncAnimation(fig, update, frames=steps, interval=interval_ms, blit=False)
plt.show()
